<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/ch14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Shakespeare Training Dataset

In [1]:
from pathlib import Path
import urllib.request

def download_shkespeare_text():
  path = Path("datasets/shakespeare/shakespeare.txt")
  if not path.is_file():
    path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://homl.info/shakespeare"
    urllib.request.urlretrieve(url, path)
  return path.read_text()

shakespeare_text = download_shkespeare_text()


In [2]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [3]:
len(shakespeare_text)

1115394

In [4]:
type(shakespeare_text)

str

In [5]:
# find the characters used in the text
vocab = sorted(set(shakespeare_text.lower()))
print(vocab)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
"".join(vocab) # this is just to visualize them "closer together"

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [7]:
# We now create the two translation dictionaries
char_to_id = {char:index for index, char in enumerate(vocab)}
id_to_char = {index:char for index, char in enumerate(vocab)}


In [8]:
# two helper functions to encode text to tensors of token IDs
import torch

def encode_text(text):
  return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
  return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [9]:
encoded = encode_text("Hello, world!")
encoded


tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])

In [10]:
decode_text(encoded)

'hello, world!'

### CharDataset Class

In [11]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
  def __init__(self, text, window_length):
    self.encoded_text =  encode_text(text)
    self.window_length = window_length

  def __len__(self): # how many valids windows
    return len(self.encoded_text) - self.window_length

  # this takes an id as argument and returns the
  # corresponding training window and target (=window shifted by 1)
  def __getitem__(self, idx):
    if idx >= len(self):
      raise IndexError("dataset index out of range")
    end = idx + self.window_length
    window = self.encoded_text[idx:end]
    target = self.encoded_text[idx+1:end+1]
    return window, target

### DataLoaders

In [12]:
window_length = 50
batch_size = 64

train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)


train_loader = DataLoader(train_set, batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size)
test_loader = DataLoader(test_set, batch_size)

## Char-RNN Model

In [13]:
if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"

In [14]:
import torch.nn as nn
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)

torch.manual_seed(42)
model = ShakespeareModel(len(vocab)).to(device)